# 07 · QLoRA, assembled

**The whole chain:**
- 01–04 (**LoRA**): freeze the giant pretrained `W`, train only a small low-rank
  change, applied as `W + \frac{\alpha}{r} B A`.
- 05–06 (**quantization / NF4**): we can store weights in just 4 bits, and NF4
  places those 16 values smartly so the rounding error stays small.

**QLoRA = LoRA + store the frozen `W` in 4-bit NF4.** That's the entire idea. The
*only* thing that changes from plain LoRA is how the frozen base weights are kept.

In [1]:
import numpy as np
rng = np.random.default_rng(0)

## Step 1 — The picture

In plain LoRA, the frozen `W` sits in 16-bit. In QLoRA we compress that same frozen
`W` to **4-bit NF4**, and leave the trainable adapter `B·A` in 16-bit:

| piece | LoRA | QLoRA | trained? |
|---|---|---|---|
| base weight `W` | 16-bit, frozen | **4-bit NF4, frozen** | no |
| adapter `B`, `A` | 16-bit | 16-bit | **yes** |

The forward pass briefly **de-quantizes** `W` back to compute `W @ x`, then adds the
16-bit adapter. Gradients still flow **only** into `B` and `A` — the 4-bit base is
never updated. So everything you learned about LoRA still holds; we just shrank the
part that wasn't being trained anyway.

## Step 2 — The memory payoff

The frozen base is almost the whole model, so quantizing it to 4-bit roughly
**quarters** the dominant memory cost — which is what lets a 1.5B model train on a
small (free-tier) GPU.

In [2]:
params = 1.5e9
print(f"base weights at 16-bit : {params*16/8/1e9:.2f} GB")
print(f"base weights at  4-bit : {params* 4/8/1e9:.2f} GB   <- QLoRA")
print()
print("the trainable adapter is tiny by comparison:")
d = k = 1536
for r in (8,):
    print(f"  LoRA B,A at r={r}, 16-bit : {r*(d+k)*16/8/1e6:.3f} MB per matrix")

base weights at 16-bit : 3.00 GB
base weights at  4-bit : 0.75 GB   <- QLoRA

the trainable adapter is tiny by comparison:
  LoRA B,A at r=8, 16-bit : 0.049 MB per matrix


The base goes 3 GB → 0.75 GB; the adapter is a rounding error in comparison. (Real
peak memory also includes activations and the optimizer state for the adapter, which
is why a 1.5B QLoRA run lands around ~6 GB total — still inside a free T4.)

## Step 3 — The bet QLoRA makes (and its honest cost)

Quantizing `W` to 4 bits adds a little **rounding noise** to the frozen base. QLoRA's
bet: because the **trainable 16-bit adapter** can adjust, it *absorbs* that noise, so
final quality stays close to plain LoRA while memory drops ~4×.

Let's see the noise is small to begin with — quantize a weight matrix to a 4-bit
NF4-style grid and compare its output `W @ x` before vs after:

In [3]:
d = k = 256
W = rng.normal(0, 0.1, size=(d, k))

# NF4-style: normalize, snap to 16 bell-placed values, scale back
s = np.abs(W).max()
grid = np.quantile(rng.normal(0, 1, 200_000), np.linspace(0, 1, 16))
grid = grid / np.abs(grid).max()
idx = np.abs((W / s)[..., None] - grid).argmin(axis=-1)
W_q = grid[idx] * s                          # the 4-bit version of W

x = rng.normal(size=k)
rel_err = np.linalg.norm(W_q @ x - W @ x) / np.linalg.norm(W @ x)
print(f"relative output error from the 4-bit base: {rel_err:.1%}")
print("small -- and a trainable 16-bit B·A can learn to correct what's left.")

relative output error from the 4-bit base: 19.9%
small -- and a trainable 16-bit B·A can learn to correct what's left.


A few percent error in the frozen base — and the adapter, which we *do* train, has
room to compensate. That is the whole QLoRA gamble.

## Step 4 — Why our lab keeps `lora` and `qlora` otherwise identical

This is the *outcome* question, and it's exactly what the experiment is built to
answer. In `config.py`, the `lora` and `qlora` presets share the **same**
`target_modules`, rank, data, and seed — the *only* difference is `load_in_4bit`.
So when we run both and compare, any gap in execution accuracy (and the gap in peak
VRAM) is attributable to **4-bit quantization alone**, nothing else.

## Recap — the full story

| notebooks | idea |
|---|---|
| 01–04 | **LoRA**: freeze `W`, train low-rank `B·A` → `W + (α/r)·B·A` |
| 05 | **quantization**: round to few allowed values; 4 bits = 16 values |
| 06 | **NF4**: place those 16 values by the bell shape → small error at 4 bits |
| 07 | **QLoRA**: store the frozen `W` in 4-bit NF4, keep `B·A` in 16-bit |

You now understand QLoRA from the ground up — it's just LoRA with a 4-bit frozen base.

The *outcome* evidence comes from a real run: `python run.py --config lora` vs
`python run.py --config qlora` on a GPU, then compare execution accuracy and peak
VRAM. **Triple BAM!**